<a href="https://colab.research.google.com/github/AndreiAf02/STAT561_Project/blob/main/STAT561_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import seaborn as sns
import pandas as pd

In [2]:
path_to_data = 'https://raw.githubusercontent.com/AndreiAf02/STAT561_Project/main/'

According to the documentation for the European Union Statistics on Income and Living Conditions (EU-SILC) given [here](https://ec.europa.eu/eurostat/documents/203647/16195750/2021_Doc65_EUSILC_User_Guide.pdf), the statistics of interest are defined as follows:
* **DB010**: Survey Year (taken to be 2013, the most recent year available)
* **DB020**: Residence Country
* **DB030**: Household ID (unique identifier)
* **DB040**: Region of Residence (used in our study for within-country stratification of the sample)
* **HB030**: Household ID (identical to DB030 above)
* **HH021**: Tenure Status (1 - *Owner w/o mortgage* ; 2 - *Owner w/ mortgage* ; 3 - *Tenant at market price* ; 4 - *Tenant at reduced price* ; 5 - *Tenant at market price*)
* **HH030**: Number of available rooms (maximum value of 10) **TO DISCUSS THAT COULD LEAD TO MEASUREMENT ERROR!**
* **HH050**: Able to warm home adequately (1 - *Yes* ; 2 - *No*)
* **HH070**: Total Housing Cost (cost per month, including mortgage interest payments, utilities, etc.)
* **HH071**: Mortgage Principal Repayment (cost per month)
* **HS050**: Afford meat (incl. fish, chicken, or vegetarian equivalent) every other day (1 - *Yes* ; 2 - *No*)
* **HS060**: Able to cover unexpected expenses (1 - *Yes* ; 2 - *No*)
* **HS120**: Able to make ends meet (1 - *great difficulty* ;  2 - *difficulty* ; 3 - *some difficulty* ; 4 - *fairly easily* ; 5 - *easily* ; 6 - *very easily*)




In this case, we define the function for calculating the sample variances for the strata in each country (where stratification is present) below:

In [131]:
def strat(data, label):
      country = []
      region = []
      size = []
      weight = []

      strat_mean = []

      strat_var = []

      Region_list = data['DB040'].unique()

      stratif = pd.DataFrame(columns=['Region', 'Size', 'Weight'])

      for i in range(len(Region_list)):
          data_1 = data[data['DB040'] == Region_list[i]]
          data_len = len(data_1)
          wt_1 = data_len/size_total

          mean_st = np.mean(data_1[label])

          # if label=='HH050' or label=='HS050' or label=='HS060':
          if label in ('HH050', 'HS050', 'HS060'):
            mean_st = 2 - np.mean(data_1[label])
            var_st = data_len/(data_len-1)*mean_st*(1-mean_st)
          else:
            mean_st = np.mean(data_1[label])
            var_st = np.var(data_1[label], ddof=1)

          country.append(data_1['DB020'].unique())
          region.append(Region_list[i])
          size.append(data_len)
          weight.append(wt_1)
          strat_mean.append(mean_st)
          strat_var.append(var_st)
      stratif = pd.DataFrame([country, region, size, weight, strat_mean, strat_var]).T
      stratif.columns =['Country', 'Region', 'Size', 'Weight', 'Mean of '+str(label), 'Var of '+str(label)]
      print(stratif)

##Austria:

In [92]:
data_AT = pd.read_csv(path_to_data + "AT_Data.csv", sep=",", header=0)
data_AT = data_AT[data_AT['DB010'].notnull()]
data_AT

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HH070,HH071,HS050,HS060,HS120
0,2013,AT,295,AT1,295,3,3,1,238.73,707.142500,1,1.0,6.0
1,2013,AT,524,AT1,524,1,2,1,501.38,NaN,1,1.0,3.0
2,2013,AT,1693,AT1,1693,3,3,1,529.67,130.952500,1,2.0,2.0
3,2013,AT,2979,AT1,2979,1,2,1,341.75,NaN,1,2.0,3.0
4,2013,AT,3173,AT1,3173,3,3,1,47.48,NaN,1,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5972,2013,AT,3699379,AT3,3699379,3,1,1,272.59,NaN,1,1.0,4.0
5973,2013,AT,3700053,AT3,3700053,3,5,1,257.67,316.666667,1,1.0,5.0
5974,2013,AT,3700232,AT3,3700232,4,5,1,790.50,533.333333,1,1.0,3.0
5975,2013,AT,3700410,AT3,3700410,3,6,1,402.94,135.341667,1,2.0,3.0


In [128]:
# data_AT.count(data_AT['HH050']==1)
count = data_AT['HH050'].value_counts()
count
# count.values()

,count
HH050,
1,5772
2,205


In [136]:
AT_result = strat(data_AT, 'HH050')
AT_result

  Country Region  Size    Weight Mean of HH050 Var of HH050
0    [AT]    AT1  2618  0.438012      0.964095     0.034629
1    [AT]    AT2  1204  0.201439      0.968439     0.030591
2    [AT]    AT3  2155  0.360549      0.966125     0.032742


##Belgium:

In [9]:
data_BE = pd.read_csv(path_to_data + "BE_Data.csv", sep=",", header=0)
data_BE = data_BE[data_BE['DB010'].notnull()]
data_BE

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,BE,22.0,BE10,22.0,3.0,6.0,1.0,1.0,1.0,5.0
1,2013.0,BE,28.0,BE10,28.0,1.0,6.0,2.0,1.0,1.0,5.0
2,2013.0,BE,37.0,BE10,37.0,3.0,5.0,1.0,2.0,1.0,3.0
3,2013.0,BE,43.0,BE10,43.0,3.0,3.0,1.0,1.0,1.0,4.0
4,2013.0,BE,50.0,BE10,50.0,1.0,6.0,1.0,1.0,1.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...
6154,2013.0,BE,95716.0,BE35,95716.0,1.0,6.0,1.0,1.0,1.0,6.0
6155,2013.0,BE,95722.0,BE35,95722.0,2.0,6.0,1.0,1.0,1.0,6.0
6156,2013.0,BE,95723.0,BE35,95723.0,2.0,5.0,1.0,1.0,1.0,2.0
6157,2013.0,BE,95821.0,BE35,95821.0,1.0,4.0,1.0,1.0,1.0,4.0


##Bulgaria:

In [10]:
data_BG = pd.read_csv(path_to_data + "BG_Data.csv", sep=",", header=0)
data_BG = data_BG[data_BG['DB010'].notnull()]
data_BG

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,BG,31.0,BG31,31.0,1.0,3.0,2.0,2.0,2.0,4.0
1,2013.0,BG,41.0,BG31,41.0,1.0,3.0,2.0,1.0,2.0,2.0
2,2013.0,BG,53.0,BG31,53.0,1.0,2.0,1.0,2.0,2.0,1.0
3,2013.0,BG,63.0,BG31,63.0,1.0,3.0,2.0,1.0,1.0,3.0
4,2013.0,BG,72.0,BG31,72.0,1.0,2.0,1.0,2.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
4966,2013.0,BG,133242.0,BG42,133242.0,1.0,2.0,1.0,1.0,2.0,3.0
4967,2013.0,BG,133246.0,BG42,133246.0,1.0,4.0,2.0,1.0,1.0,2.0
4968,2013.0,BG,133249.0,BG42,133249.0,1.0,4.0,2.0,1.0,1.0,5.0
4969,2013.0,BG,133251.0,BG42,133251.0,1.0,6.0,1.0,2.0,2.0,4.0


##Cyprus:

In [11]:
data_CY = pd.read_csv(path_to_data + "CY_Data.csv", sep=",", header=0)
data_CY = data_CY[data_CY['DB010'].notnull()]
data_CY

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,CY,4.0,CY00,4.0,3.0,6.0,1.0,1.0,2.0,1.0
1,2013.0,CY,25.0,CY00,25.0,2.0,6.0,1.0,2.0,1.0,2.0
2,2013.0,CY,41.0,CY00,41.0,3.0,6.0,1.0,1.0,1.0,1.0
3,2013.0,CY,49.0,CY00,49.0,1.0,5.0,2.0,1.0,2.0,1.0
4,2013.0,CY,54.0,CY00,54.0,1.0,2.0,1.0,2.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
4643,2013.0,CY,62043.0,CY00,62043.0,2.0,4.0,1.0,1.0,1.0,4.0
4644,2013.0,CY,62075.0,CY00,62075.0,2.0,6.0,1.0,1.0,2.0,1.0
4645,2013.0,CY,62129.0,CY00,62129.0,1.0,4.0,1.0,1.0,1.0,6.0
4646,2013.0,CY,62147.0,CY00,62147.0,1.0,6.0,1.0,1.0,1.0,6.0


##Germany:

In [12]:
data_DE = pd.read_csv(path_to_data + "DE_Data.csv", sep=",", header=0)
data_DE = data_DE[data_DE['DB010'].notnull()]
data_DE

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,DE,356.0,NaN,356.0,5.0,3.0,1.0,1.0,1.0,5.0
1,2013.0,DE,447.0,NaN,447.0,1.0,6.0,1.0,1.0,2.0,5.0
2,2013.0,DE,599.0,NaN,599.0,3.0,4.0,1.0,1.0,1.0,5.0
3,2013.0,DE,711.0,NaN,711.0,2.0,6.0,1.0,2.0,2.0,4.0
4,2013.0,DE,795.0,NaN,795.0,2.0,5.0,1.0,1.0,1.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...
12698,2013.0,DE,4010281.0,NaN,4010281.0,1.0,3.0,1.0,1.0,2.0,3.0
12699,2013.0,DE,4010304.0,NaN,4010304.0,2.0,5.0,1.0,1.0,1.0,3.0
12700,2013.0,DE,4010309.0,NaN,4010309.0,3.0,3.0,1.0,1.0,1.0,2.0
12701,2013.0,DE,4010422.0,NaN,4010422.0,2.0,3.0,2.0,1.0,1.0,5.0


##Denmark:

In [15]:
data_DK = pd.read_csv(path_to_data + "DK_Data.csv", sep=",", header=0)
data_DK = data_DK[data_DK['DB010'].notnull()]
data_DK

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,DK,83.0,DK01,83.0,1.0,6.0,1.0,1.0,1.0,5.0
1,2013.0,DK,105.0,DK01,105.0,1.0,6.0,1.0,1.0,1.0,6.0
2,2013.0,DK,139.0,DK01,139.0,2.0,6.0,1.0,1.0,1.0,4.0
3,2013.0,DK,162.0,DK01,162.0,2.0,6.0,1.0,1.0,1.0,4.0
4,2013.0,DK,165.0,DK01,165.0,2.0,6.0,1.0,1.0,1.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...
5414,2013.0,DK,140226.0,DK05,140226.0,2.0,2.0,1.0,1.0,1.0,6.0
5415,2013.0,DK,140311.0,DK05,140311.0,1.0,6.0,1.0,1.0,1.0,6.0
5416,2013.0,DK,140321.0,DK05,140321.0,1.0,6.0,1.0,1.0,1.0,4.0
5417,2013.0,DK,140332.0,DK05,140332.0,3.0,2.0,1.0,1.0,2.0,4.0


##Estonia:

In [17]:
data_EE = pd.read_csv(path_to_data + "EE_Data.csv", sep=",", header=0)
data_EE = data_EE[data_EE['DB010'].notnull()]
data_EE

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,EE,7.0,EE00,7.0,5.0,5.0,1.0,1.0,2.0,3.0
1,2013.0,EE,24.0,EE00,24.0,1.0,6.0,1.0,1.0,2.0,2.0
2,2013.0,EE,49.0,EE00,49.0,2.0,3.0,1.0,1.0,1.0,5.0
3,2013.0,EE,52.0,EE00,52.0,2.0,6.0,1.0,1.0,1.0,5.0
4,2013.0,EE,79.0,EE00,79.0,5.0,2.0,1.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
5770,2013.0,EE,117876.0,EE00,117876.0,2.0,3.0,1.0,1.0,1.0,5.0
5771,2013.0,EE,117880.0,EE00,117880.0,2.0,6.0,1.0,1.0,2.0,3.0
5772,2013.0,EE,117932.0,EE00,117932.0,1.0,6.0,1.0,1.0,2.0,4.0
5773,2013.0,EE,117954.0,EE00,117954.0,2.0,4.0,1.0,1.0,1.0,4.0


##Greece:

In [18]:
data_EL = pd.read_csv(path_to_data + "EL_Data.csv", sep=",", header=0)
data_EL = data_EL[data_EL['DB010'].notnull()]
data_EL

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,EL,60.0,EL30,60.0,5.0,3.0,1.0,1.0,1.0,1.0
1,2013.0,EL,65.0,EL30,65.0,1.0,4.0,1.0,1.0,1.0,1.0
2,2013.0,EL,82.0,EL30,82.0,1.0,4.0,2.0,1.0,1.0,3.0
3,2013.0,EL,109.0,EL30,109.0,3.0,5.0,1.0,1.0,1.0,3.0
4,2013.0,EL,122.0,EL30,122.0,1.0,3.0,1.0,2.0,1.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...
7434,2013.0,EL,84911.0,EL65,84911.0,1.0,4.0,2.0,1.0,2.0,1.0
7435,2013.0,EL,84912.0,EL65,84912.0,1.0,4.0,1.0,1.0,1.0,1.0
7436,2013.0,EL,84914.0,EL65,84914.0,1.0,5.0,1.0,1.0,1.0,6.0
7437,2013.0,EL,84924.0,EL65,84924.0,1.0,5.0,1.0,1.0,1.0,5.0


##Spain:

In [19]:
data_ES = pd.read_csv(path_to_data + "ES_Data.csv", sep=",", header=0)
data_ES = data_ES[data_ES['DB010'].notnull()]
data_ES

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,ES,43.0,ES11,43.0,1.0,6.0,1.0,1.0,1.0,4.0
1,2013.0,ES,55.0,ES11,55.0,3.0,4.0,1.0,1.0,2.0,4.0
2,2013.0,ES,72.0,ES11,72.0,1.0,6.0,1.0,1.0,1.0,3.0
3,2013.0,ES,85.0,ES11,85.0,4.0,6.0,1.0,1.0,2.0,5.0
4,2013.0,ES,97.0,ES11,97.0,2.0,4.0,1.0,1.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
12134,2013.0,ES,363504.0,ES70,363504.0,1.0,6.0,1.0,1.0,1.0,4.0
12135,2013.0,ES,363592.0,ES70,363592.0,3.0,5.0,1.0,1.0,1.0,6.0
12136,2013.0,ES,363713.0,ES70,363713.0,2.0,3.0,1.0,1.0,2.0,3.0
12137,2013.0,ES,363723.0,ES70,363723.0,3.0,4.0,1.0,1.0,1.0,5.0


##Finland:

In [20]:
data_FI = pd.read_csv(path_to_data + "FI_Data.csv", sep=",", header=0)
data_FI = data_FI[data_FI['DB010'].notnull()]
data_FI

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,FI,142.0,FI19,142.0,3.0,1.0,1.0,1.0,1.0,5.0
1,2013.0,FI,300.0,FI19,300.0,1.0,2.0,2.0,1.0,2.0,6.0
2,2013.0,FI,315.0,FI19,315.0,2.0,3.0,1.0,1.0,1.0,4.0
3,2013.0,FI,354.0,FI19,354.0,2.0,5.0,1.0,1.0,1.0,6.0
4,2013.0,FI,862.0,FI19,862.0,2.0,4.0,1.0,1.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
11365,2013.0,FI,2593649.0,FI1D,2593649.0,4.0,5.0,1.0,1.0,1.0,6.0
11366,2013.0,FI,2594037.0,FI1D,2594037.0,2.0,4.0,1.0,1.0,2.0,6.0
11367,2013.0,FI,2594214.0,FI1D,2594214.0,2.0,2.0,1.0,1.0,1.0,5.0
11368,2013.0,FI,2594542.0,FI1D,2594542.0,1.0,3.0,1.0,1.0,1.0,6.0


##France:

In [21]:
data_FR = pd.read_csv(path_to_data + "FR_Data.csv", sep=",", header=0)
data_FR = data_FR[data_FR['DB010'].notnull()]
data_FR

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,FR,211.0,FR10,211.0,2.0,5.0,1.0,1.0,1.0,3.0
1,2013.0,FR,265.0,FR10,265.0,2.0,6.0,1.0,1.0,1.0,2.0
2,2013.0,FR,354.0,FR10,354.0,3.0,6.0,1.0,1.0,2.0,4.0
3,2013.0,FR,420.0,FR10,420.0,2.0,4.0,1.0,1.0,1.0,4.0
4,2013.0,FR,471.0,FR10,471.0,5.0,3.0,1.0,1.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...
11126,2013.0,FR,557144.0,FR83,557144.0,1.0,4.0,1.0,1.0,1.0,3.0
11127,2013.0,FR,557146.0,FR83,557146.0,1.0,2.0,1.0,1.0,1.0,5.0
11128,2013.0,FR,557162.0,FR83,557162.0,2.0,3.0,1.0,1.0,2.0,3.0
11129,2013.0,FR,557197.0,FR83,557197.0,3.0,4.0,1.0,1.0,1.0,4.0


##Croatia:

In [22]:
data_HR = pd.read_csv(path_to_data + "HR_Data.csv", sep=",", header=0)
data_HR = data_HR[data_HR['DB010'].notnull()]
data_HR

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,HR,52.0,HR03,52.0,5.0,3.0,2.0,1.0,2.0,1.0
1,2013.0,HR,66.0,HR03,66.0,1.0,6.0,1.0,1.0,1.0,1.0
2,2013.0,HR,87.0,HR03,87.0,1.0,2.0,1.0,1.0,1.0,3.0
3,2013.0,HR,100.0,HR03,100.0,1.0,5.0,1.0,1.0,2.0,3.0
4,2013.0,HR,103.0,HR03,103.0,1.0,3.0,1.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
5360,2013.0,HR,75894.0,HR04,75894.0,4.0,4.0,1.0,1.0,2.0,3.0
5361,2013.0,HR,75906.0,HR04,75906.0,1.0,4.0,1.0,1.0,2.0,3.0
5362,2013.0,HR,75914.0,HR04,75914.0,1.0,3.0,2.0,2.0,1.0,2.0
5363,2013.0,HR,75920.0,HR04,75920.0,NaN,NaN,NaN,NaN,NaN,NaN


##Hungary:

In [23]:
data_HU = pd.read_csv(path_to_data + "HU_Data.csv", sep=",", header=0)
data_HU = data_HU[data_HU['DB010'].notnull()]
data_HU

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,HU,115.0,HU1,115.0,1.0,2.0,1.0,1.0,2.0,2.0
1,2013.0,HU,121.0,HU1,121.0,1.0,2.0,1.0,1.0,2.0,1.0
2,2013.0,HU,152.0,HU1,152.0,1.0,3.0,2.0,1.0,2.0,1.0
3,2013.0,HU,203.0,HU1,203.0,1.0,3.0,1.0,1.0,2.0,3.0
4,2013.0,HU,234.0,HU1,234.0,1.0,3.0,1.0,1.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
10218,2013.0,HU,188351.0,HU3,188351.0,2.0,3.0,1.0,1.0,2.0,4.0
10219,2013.0,HU,188362.0,HU3,188362.0,5.0,3.0,1.0,1.0,2.0,2.0
10220,2013.0,HU,188397.0,HU3,188397.0,1.0,6.0,1.0,1.0,2.0,1.0
10221,2013.0,HU,188406.0,HU3,188406.0,2.0,3.0,1.0,2.0,1.0,1.0


##Ireland:

In [24]:
data_IE = pd.read_csv(path_to_data + "IE_Data.csv", sep=",", header=0)
data_IE = data_IE[data_IE['DB010'].notnull()]
data_IE

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,IE,47.0,IE01,47.0,3.0,4.0,2.0,1.0,1.0,4.0
1,2013.0,IE,60.0,IE01,60.0,2.0,5.0,1.0,1.0,2.0,4.0
2,2013.0,IE,79.0,IE01,79.0,1.0,4.0,1.0,1.0,2.0,2.0
3,2013.0,IE,93.0,IE01,93.0,3.0,6.0,1.0,1.0,2.0,2.0
4,2013.0,IE,106.0,IE01,106.0,1.0,6.0,2.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
4917,2013.0,IE,85348.0,IE02,85348.0,1.0,6.0,1.0,1.0,2.0,1.0
4918,2013.0,IE,85379.0,IE02,85379.0,3.0,5.0,1.0,1.0,2.0,1.0
4919,2013.0,IE,85440.0,IE02,85440.0,2.0,5.0,1.0,1.0,1.0,3.0
4920,2013.0,IE,85453.0,IE02,85453.0,2.0,6.0,1.0,1.0,2.0,6.0


#Italy:

In [37]:
data_IT1 = pd.read_csv(path_to_data + "IT_Data_Pt1.csv", sep=",", header=0)
data_IT1 = data_IT1[data_IT1['DB010'].notnull()]

data_IT2 = pd.read_csv(path_to_data + "IT_Data_Pt2.csv", sep=",", header=0)
data_IT2 = data_IT2[data_IT2['DB010'].notnull()]
data_IT2 = data_IT2[data_IT2['DB010'] != 0]

data_IT = pd.concat([data_IT1, data_IT2])
data_IT.reset_index()


,index,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,0,2013.0,IT,82.0,ITC1,82.0,1.0,5.0,1.0,1.0,1.0,4.0
1,1,2013.0,IT,103.0,ITC1,103.0,1.0,5.0,1.0,1.0,1.0,4.0
2,2,2013.0,IT,137.0,ITC1,137.0,5.0,5.0,1.0,1.0,1.0,3.0
3,3,2013.0,IT,163.0,ITC1,163.0,1.0,4.0,1.0,1.0,1.0,2.0
4,4,2013.0,IT,183.0,ITC1,183.0,3.0,3.0,1.0,1.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
18482,8482,2013.0,IT,510956.0,ITI4,510956.0,1.0,5.0,1.0,1.0,1.0,3.0
18483,8483,2013.0,IT,511021.0,ITI4,511021.0,1.0,5.0,1.0,1.0,1.0,6.0
18484,8484,2013.0,IT,511059.0,ITI4,511059.0,1.0,2.0,1.0,1.0,1.0,3.0
18485,8485,2013.0,IT,511078.0,ITI4,511078.0,3.0,5.0,1.0,1.0,1.0,3.0


##Lithuania:

In [38]:
data_LT = pd.read_csv(path_to_data + "LT_Data.csv", sep=",", header=0)
data_LT = data_LT[data_LT['DB010'].notnull()]
data_LT

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,LT,7.0,LT00,7.0,1.0,5.0,1.0,2.0,1.0,2.0
1,2013.0,LT,54.0,LT00,54.0,1.0,3.0,2.0,1.0,1.0,4.0
2,2013.0,LT,58.0,LT00,58.0,1.0,4.0,1.0,1.0,2.0,3.0
3,2013.0,LT,88.0,LT00,88.0,2.0,4.0,1.0,1.0,1.0,2.0
4,2013.0,LT,104.0,LT00,104.0,1.0,4.0,1.0,1.0,2.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
5137,2013.0,LT,130113.0,LT00,130113.0,1.0,4.0,1.0,1.0,2.0,4.0
5138,2013.0,LT,130279.0,LT00,130279.0,4.0,1.0,1.0,2.0,1.0,4.0
5139,2013.0,LT,130294.0,LT00,130294.0,1.0,1.0,1.0,1.0,1.0,4.0
5140,2013.0,LT,130373.0,LT00,130373.0,1.0,4.0,1.0,1.0,1.0,2.0


##Luxembourg:

In [39]:
data_LU = pd.read_csv(path_to_data + "LU_Data.csv", sep=",", header=0)
data_LU = data_LU[data_LU['DB010'].notnull()]
data_LU

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,LU,3.0,LU00,3.0,1.0,5.0,1.0,1.0,1.0,5.0
1,2013.0,LU,16.0,LU00,16.0,1.0,4.0,1.0,1.0,1.0,6.0
2,2013.0,LU,27.0,LU00,27.0,1.0,5.0,1.0,1.0,1.0,5.0
3,2013.0,LU,31.0,LU00,31.0,1.0,4.0,1.0,1.0,1.0,1.0
4,2013.0,LU,35.0,LU00,35.0,1.0,6.0,1.0,1.0,1.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...
3765,2013.0,LU,40975.0,LU00,40975.0,1.0,5.0,1.0,1.0,1.0,6.0
3766,2013.0,LU,40979.0,LU00,40979.0,3.0,6.0,2.0,1.0,2.0,6.0
3767,2013.0,LU,41031.0,LU00,41031.0,1.0,6.0,1.0,1.0,1.0,5.0
3768,2013.0,LU,41032.0,LU00,41032.0,1.0,6.0,1.0,1.0,1.0,6.0


##Latvia:

In [40]:
data_LV = pd.read_csv(path_to_data + "LV_Data.csv", sep=",", header=0)
data_LV = data_LV[data_LV['DB010'].notnull()]
data_LV

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,LV,5.0,LV00,5.0,1.0,3.0,1.0,2.0,2.0,2.0
1,2013.0,LV,17.0,LV00,17.0,2.0,3.0,2.0,1.0,1.0,4.0
2,2013.0,LV,34.0,LV00,34.0,1.0,5.0,1.0,1.0,2.0,1.0
3,2013.0,LV,36.0,LV00,36.0,1.0,1.0,1.0,2.0,2.0,3.0
4,2013.0,LV,42.0,LV00,42.0,3.0,3.0,1.0,1.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
6304,2013.0,LV,83007.0,LV00,83007.0,2.0,6.0,1.0,2.0,2.0,4.0
6305,2013.0,LV,83012.0,LV00,83012.0,1.0,6.0,1.0,2.0,1.0,2.0
6306,2013.0,LV,83027.0,LV00,83027.0,2.0,4.0,1.0,1.0,2.0,1.0
6307,2013.0,LV,83040.0,LV00,83040.0,3.0,3.0,1.0,2.0,1.0,4.0


##Malta:

In [41]:
data_MT = pd.read_csv(path_to_data + "MT_Data.csv", sep=",", header=0)
data_MT = data_MT[data_MT['DB010'].notnull()]
data_MT

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,MT,2.0,MT00,2.0,4.0,5.0,1.0,1.0,1.0,4.0
1,2013.0,MT,12.0,MT00,12.0,1.0,4.0,1.0,1.0,2.0,1.0
2,2013.0,MT,20.0,MT00,20.0,1.0,5.0,1.0,1.0,2.0,1.0
3,2013.0,MT,24.0,MT00,24.0,1.0,6.0,2.0,1.0,1.0,3.0
4,2013.0,MT,26.0,MT00,26.0,1.0,6.0,1.0,1.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...
4376,2013.0,MT,31186.0,MT00,31186.0,1.0,5.0,1.0,1.0,1.0,5.0
4377,2013.0,MT,31193.0,MT00,31193.0,3.0,6.0,2.0,1.0,2.0,2.0
4378,2013.0,MT,31195.0,MT00,31195.0,1.0,6.0,1.0,1.0,1.0,3.0
4379,2013.0,MT,31213.0,MT00,31213.0,1.0,6.0,1.0,1.0,1.0,4.0


##Netherlands:

In [42]:
data_NL = pd.read_csv(path_to_data + "NL_Data.csv", sep=",", header=0)
data_NL = data_NL[data_NL['DB010'].notnull()]
data_NL

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,NL,15.0,NaN,15.0,2.0,6.0,1.0,1.0,1.0,4.0
1,2013.0,NL,26.0,NaN,26.0,3.0,4.0,1.0,1.0,2.0,3.0
2,2013.0,NL,83.0,NaN,83.0,2.0,5.0,2.0,1.0,1.0,3.0
3,2013.0,NL,124.0,NaN,124.0,2.0,5.0,1.0,1.0,1.0,3.0
4,2013.0,NL,146.0,NaN,146.0,3.0,6.0,1.0,1.0,1.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...
10126,2013.0,NL,151346.0,NaN,151346.0,3.0,6.0,1.0,1.0,1.0,5.0
10127,2013.0,NL,151353.0,NaN,151353.0,1.0,5.0,1.0,1.0,1.0,4.0
10128,2013.0,NL,151357.0,NaN,151357.0,3.0,4.0,1.0,1.0,1.0,4.0
10129,2013.0,NL,151380.0,NaN,151380.0,3.0,5.0,1.0,1.0,2.0,5.0


##Romania:

In [7]:
data_RO = pd.read_csv(path_to_data + "RO_Data.csv", sep=",", header=0)
data_RO = data_RO[data_RO['DB010'].notnull()]
data_RO

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,RO,41.0,RO11,41.0,1.0,4.0,1.0,1.0,1.0,4.0
1,2013.0,RO,52.0,RO11,52.0,5.0,4.0,1.0,1.0,1.0,3.0
2,2013.0,RO,68.0,RO11,68.0,1.0,4.0,1.0,1.0,1.0,2.0
3,2013.0,RO,81.0,RO11,81.0,1.0,5.0,1.0,1.0,1.0,4.0
4,2013.0,RO,92.0,RO11,92.0,1.0,3.0,1.0,1.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
7555,2013.0,RO,140322.0,RO42,140322.0,3.0,2.0,2.0,2.0,2.0,1.0
7556,2013.0,RO,140353.0,RO42,140353.0,1.0,3.0,1.0,1.0,1.0,1.0
7557,2013.0,RO,140364.0,RO42,140364.0,5.0,2.0,1.0,1.0,1.0,4.0
7558,2013.0,RO,140369.0,RO42,140369.0,1.0,3.0,1.0,2.0,2.0,1.0


##Sweden:

In [43]:
data_SE = pd.read_csv(path_to_data + "SE_Data.csv", sep=",", header=0)
data_SE = data_SE[data_SE['DB010'].notnull()]
data_SE

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,SE,39.0,SE11,39.0,2.0,5.0,1.0,1.0,1.0,6.0
1,2013.0,SE,49.0,SE11,49.0,2.0,3.0,1.0,1.0,1.0,4.0
2,2013.0,SE,65.0,SE11,65.0,2.0,5.0,1.0,1.0,1.0,4.0
3,2013.0,SE,77.0,SE11,77.0,2.0,5.0,1.0,1.0,2.0,6.0
4,2013.0,SE,88.0,SE11,88.0,2.0,4.0,1.0,1.0,1.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...
6196,2013.0,SE,92654.0,SE33,92654.0,2.0,5.0,1.0,1.0,1.0,4.0
6197,2013.0,SE,92655.0,SE33,92655.0,2.0,4.0,1.0,1.0,1.0,6.0
6198,2013.0,SE,92659.0,SE33,92659.0,2.0,4.0,1.0,1.0,1.0,6.0
6199,2013.0,SE,92684.0,SE33,92684.0,2.0,5.0,1.0,1.0,1.0,6.0


##Slovenia:

In [44]:
data_SI = pd.read_csv(path_to_data + "SI_Data.csv", sep=",", header=0)
data_SI = data_SI[data_SI['DB010'].notnull()]
data_SI

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,SI,486.0,NaN,486.0,1.0,3.0,2.0,1.0,2.0,3.0
1,2013.0,SI,611.0,NaN,611.0,1.0,3.0,1.0,1.0,2.0,1.0
2,2013.0,SI,818.0,NaN,818.0,4.0,4.0,1.0,1.0,2.0,5.0
3,2013.0,SI,971.0,NaN,971.0,1.0,5.0,1.0,1.0,2.0,5.0
4,2013.0,SI,1085.0,NaN,1085.0,1.0,4.0,1.0,2.0,2.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
8982,2013.0,SI,803675.0,NaN,803675.0,2.0,6.0,1.0,1.0,2.0,2.0
8983,2013.0,SI,803680.0,NaN,803680.0,5.0,3.0,1.0,1.0,2.0,2.0
8984,2013.0,SI,803698.0,NaN,803698.0,1.0,3.0,1.0,1.0,1.0,1.0
8985,2013.0,SI,803718.0,NaN,803718.0,5.0,4.0,1.0,1.0,2.0,2.0


##Slovakia:

In [45]:
data_SK = pd.read_csv(path_to_data + "SK_Data.csv", sep=",", header=0)
data_SK = data_SK[data_SK['DB010'].notnull()]
data_SK

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,SK,32.0,SK01,32.0,5.0,2.0,1.0,1.0,2.0,3.0
1,2013.0,SK,41.0,SK01,41.0,3.0,3.0,1.0,1.0,1.0,3.0
2,2013.0,SK,49.0,SK01,49.0,3.0,4.0,1.0,1.0,1.0,2.0
3,2013.0,SK,56.0,SK01,56.0,3.0,2.0,1.0,1.0,2.0,3.0
4,2013.0,SK,61.0,SK01,61.0,1.0,2.0,1.0,2.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
5397,2013.0,SK,92495.0,SK04,92495.0,NaN,NaN,NaN,NaN,NaN,NaN
5398,2013.0,SK,92506.0,SK04,92506.0,NaN,NaN,NaN,NaN,NaN,NaN
5399,2013.0,SK,92536.0,SK04,92536.0,NaN,NaN,NaN,NaN,NaN,NaN
5400,2013.0,SK,92550.0,SK04,92550.0,NaN,NaN,NaN,NaN,NaN,NaN


##United Kingdom:

In [46]:
data_UK = pd.read_csv(path_to_data + "UK_Data.csv", sep=",", header=0)
data_UK = data_UK[data_UK['DB010'].notnull()]
data_UK

,DB010,DB020,DB030,DB040,HB030,HH021,HH030,HH050,HS050,HS060,HS120
0,2013.0,UK,31.0,UKC1,31.0,1.0,5.0,1.0,2.0,1.0,3.0
1,2013.0,UK,56.0,UKC1,56.0,4.0,5.0,1.0,1.0,1.0,5.0
2,2013.0,UK,177.0,UKC1,177.0,3.0,4.0,1.0,1.0,2.0,3.0
3,2013.0,UK,312.0,UKC1,312.0,3.0,6.0,1.0,1.0,1.0,5.0
4,2013.0,UK,333.0,UKC1,333.0,1.0,4.0,1.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
10169,2013.0,UK,539994.0,UKN0,539994.0,1.0,5.0,1.0,1.0,2.0,4.0
10170,2013.0,UK,540001.0,UKN0,540001.0,2.0,5.0,1.0,1.0,2.0,4.0
10171,2013.0,UK,540005.0,UKN0,540005.0,3.0,2.0,1.0,1.0,1.0,2.0
10172,2013.0,UK,540009.0,UKN0,540009.0,4.0,5.0,1.0,2.0,2.0,3.0
